# Biblical Gemma 4 12B DPO Training with Unsloth (4-bit QLoRA)

Phase 2 DPO training on top of the SFT LoRA produced by `biblical_gemma4_12b_unsloth_4bit.ipynb`.

- Base model: `unsloth/gemma-4-12b-it`
- SFT LoRA input: `output/biblical_gemma4_12b_unsloth_4bit_mobile/lora_adapters`
- DPO data: `data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.jsonl`
- DPO LoRA output: `output/biblical_gemma4_12b_unsloth_4bit_mobile_dpo/lora_adapters`

Run this only after the Gemma 4 12B SFT notebook has finished and saved its LoRA adapters.

## 1. Environment Preparation

Install Unsloth and updated HuggingFace libraries (same env-prep as the SFT notebook — Gemma 4 support requires `transformers`/`peft` from git main).

In [ ]:
# Install core packages in the running notebook container
!pip install -q -U unsloth trl accelerate datasets bitsandbytes

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel ≥0.16 on PyPI, so
# uninstall — peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
!pip uninstall -y -q torchao

# Gemma 4 support may be ahead of PyPI releases.
!pip install -q -U git+https://github.com/huggingface/transformers.git

# Keep PEFT compatible with latest Transformers main.
!pip install -q -U git+https://github.com/huggingface/peft.git

# Verify installations
import importlib.util
import unsloth
import transformers
import peft
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ PEFT: {peft.__version__}")
print(f"✓ TRL: {trl.__version__}")
print(f"✓ torchao installed: {importlib.util.find_spec('torchao') is not None} (should be False)")
print("Environment ready. Restart kernel, then rerun from Cell 2 (Setup).")

## 2. Setup

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and broken
# torch.compile path. Without these, Gemma falls back to slow eager Python loops.
# Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.5,max_split_size_mb:256"

def _pip(*args):
    result = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-1000:] if result.stderr else result.stdout[-1000:])
        raise RuntimeError(f"pip failed: {' '.join(args)}")

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available. Run this in the unsloth-notebook container.")

_MEMORY_FRACTION = 0.55
torch.cuda.set_per_process_memory_fraction(_MEMORY_FRACTION, 0)

for module, install_args in {
    "psutil": ["install", "-q", "psutil"],
    "matplotlib": ["install", "-q", "matplotlib"],
    "ipywidgets": ["install", "-q", "ipywidgets"],
}.items():
    if _check_import(module) is None:
        print(f"Installing {install_args[-1]}...")
        _pip(*install_args)

# Environment Preparation requires a kernel restart. Import Unsloth first in the
# fresh process so its Transformers and PEFT patches share one module generation.
import unsloth
import transformers

if os.path.exists("/workspace/training/biblical"):
    PROJECT_ROOT = Path("/workspace/training/biblical")
elif os.path.exists("/workspace/biblical"):
    PROJECT_ROOT = Path("/workspace/biblical")
else:
    PROJECT_ROOT = Path("/home/spark/projects/training/biblical")

BASE_LLM = "unsloth/gemma-4-12b-it"
# Must match the base in the SFT adapter at SFT_LORA_PATH.
# Source SFT notebook: biblical_gemma4_12b_unsloth_4bit.ipynb
SFT_MODEL_NAME_BASE = "biblical_gemma4_12b_unsloth_4bit_mobile"
MODEL_NAME_BASE = "biblical_gemma4_12b_unsloth_4bit_mobile_dpo"
DPO_DATA_FILE = PROJECT_ROOT / "data" / "training-data" / "biblical_persona_v2" / "biblical_personas_v2_dpo.jsonl"
SFT_LORA_PATH = PROJECT_ROOT / "output" / SFT_MODEL_NAME_BASE / "lora_adapters"
OUTPUT_BASE = PROJECT_ROOT / "output" / MODEL_NAME_BASE
TRAIN_DIR = OUTPUT_BASE / "train"
LORA_OUTPUT_DIR = OUTPUT_BASE / "lora_adapters"

MAX_SEQ_LENGTH = 4096
MAX_PROMPT_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 5e-6
DPO_BETA = 0.05
TARGET_EPOCHS = 1
WARMUP_RATIO = 0.1
SAVE_STEPS = 50
LOSS_TYPE = "sigmoid"
DPO_MAX_PAIRS = 0

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch: {torch.__version__}; transformers: {transformers.__version__}")
print(f"CUDA memory fraction: {_MEMORY_FRACTION}")
print(f"Project root: {PROJECT_ROOT}")
print(f"DPO data: {DPO_DATA_FILE}")
print(f"SFT LoRA input: {SFT_LORA_PATH}")
print(f"DPO LoRA output: {LORA_OUTPUT_DIR}")
print(f"Save steps: {SAVE_STEPS}")
for path, label in [(DPO_DATA_FILE, "DPO data"), (SFT_LORA_PATH, "SFT LoRA")]:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

## 3. Load Model And Format DPO Dataset Cache

In [ ]:
import hashlib
import json
import os
import random
import shutil
from collections import Counter, defaultdict
from datasets import Dataset as HFDataset, load_from_disk
from peft import PeftModel
from unsloth import FastLanguageModel

# Load the base model without 4-bit quantization as a workaround for the
# Gemma4/Transformers quantization_config type error, then attach the saved SFT LoRA
# adapter directly via PeftModel. This keeps the adapter trainable while bypassing
# both the 4-bit base-load failure and Unsloth PEFT-adapter auto-reload path.
# The failing path raises "TypeError: Found quant_method=bitsandbytes but quantization_config
# is not a BitsAndBytesConfig" in this runtime.
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
# Gemma 4 can return a Processor object instead of a plain tokenizer.
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer
if tokenizer.pad_token is None or tokenizer.pad_token_id != tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

model = PeftModel.from_pretrained(model, str(SFT_LORA_PATH), is_trainable=True)

FORMATTED_CACHE_DIR = TRAIN_DIR / "formatted_dpo_dataset_cache"
FORMATTED_DATASET_DIR = FORMATTED_CACHE_DIR / "dataset"
FORMATTED_MANIFEST = FORMATTED_CACHE_DIR / "manifest.json"
FORMATTED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

format_config = {
    "cache_version": 1,
    "dpo_data_file": str(DPO_DATA_FILE),
    "dpo_data_mtime_ns": DPO_DATA_FILE.stat().st_mtime_ns,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "dpo_max_pairs": DPO_MAX_PAIRS,
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
format_fingerprint = hashlib.sha256(json.dumps(format_config, sort_keys=True).encode("utf-8")).hexdigest()
manifest = json.load(open(FORMATTED_MANIFEST)) if FORMATTED_MANIFEST.exists() else None

if manifest and manifest.get("fingerprint") == format_fingerprint and FORMATTED_DATASET_DIR.exists():
    dpo_dataset = load_from_disk(str(FORMATTED_DATASET_DIR))
    print(f"Loaded formatted dataset cache: {FORMATTED_DATASET_DIR}")
else:
    with open(DPO_DATA_FILE) as f:
        raw_pairs = [json.loads(line) for line in f]

    print(f"Loaded {len(raw_pairs):,} DPO pairs")
    for source, count in Counter(p.get("source", "unknown") for p in raw_pairs).most_common():
        print(f"  {source:<24} {count:>5}")

    formatted_pairs = []
    errors = []
    skipped_prompt_too_long = 0
    skipped_too_long = 0
    for index, pair in enumerate(raw_pairs):
        chosen_msgs = pair.get("chosen", [])
        rejected_msgs = pair.get("rejected", [])
        if len(chosen_msgs) != 3 or len(rejected_msgs) != 3:
            errors.append(f"pair {index}: expected three chosen/rejected messages")
            continue
        if chosen_msgs[:-1] != rejected_msgs[:-1]:
            errors.append(f"pair {index}: prompt messages differ")
            continue
        prompt = tokenizer.apply_chat_template(chosen_msgs[:-1], tokenize=False, add_generation_prompt=True)
        chosen = chosen_msgs[-1]["content"]
        rejected = rejected_msgs[-1]["content"]
        prompt_tokens = len(tokenizer.encode(prompt, add_special_tokens=False))
        chosen_tokens = len(tokenizer.encode(chosen, add_special_tokens=False))
        rejected_tokens = len(tokenizer.encode(rejected, add_special_tokens=False))
        if prompt_tokens > MAX_PROMPT_LENGTH:
            skipped_prompt_too_long += 1
            continue
        if prompt_tokens + max(chosen_tokens, rejected_tokens) > MAX_SEQ_LENGTH:
            skipped_too_long += 1
            continue
        formatted_pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected, "source": pair.get("source", "unknown"), "persona": pair.get("persona", "unknown")})

    if errors:
        print(f"Validation errors: {len(errors)}")
        for error in errors[:10]:
            print(error)
    if not formatted_pairs:
        raise RuntimeError("No DPO pairs remain after validation/filtering.")

    if DPO_MAX_PAIRS and len(formatted_pairs) > DPO_MAX_PAIRS:
        random.seed(3407)
        by_source = defaultdict(list)
        for pair in formatted_pairs:
            by_source[pair.get("source", "unknown")].append(pair)
        sampled = []
        total = len(formatted_pairs)
        for items in by_source.values():
            sample_count = max(1, round(DPO_MAX_PAIRS * len(items) / total))
            sampled.extend(random.sample(items, min(sample_count, len(items))))
        random.shuffle(sampled)
        formatted_pairs = sampled[:DPO_MAX_PAIRS]
        print(f"Capped formatted DPO dataset to {len(formatted_pairs):,} pairs")

    dpo_dataset = HFDataset.from_list(formatted_pairs).shuffle(seed=3407)
    if FORMATTED_DATASET_DIR.exists():
        shutil.rmtree(FORMATTED_DATASET_DIR)
    dpo_dataset.save_to_disk(str(FORMATTED_DATASET_DIR))
    tmp_manifest = FORMATTED_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_manifest, "w") as f:
        json.dump({"status": "complete", "fingerprint": format_fingerprint, "config": format_config, "rows": len(dpo_dataset)}, f, indent=2)
    os.replace(tmp_manifest, FORMATTED_MANIFEST)
    print(f"Saved formatted dataset cache: {FORMATTED_DATASET_DIR}")
    print(f"Filtered for prompt length: {skipped_prompt_too_long:,}")
    print(f"Filtered for total length: {skipped_too_long:,}")

print(f"DPO rows: {len(dpo_dataset):,}")


## 4. Configure DPO Trainer

In [ ]:
from trl import DPOTrainer, DPOConfig
import math, torch

FastLanguageModel.for_training(model)
effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = math.ceil(len(dpo_dataset) / effective_batch)
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, int(max_steps * WARMUP_RATIO))
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# TRL inspects model_type to route between text and vision pipelines.
# If this model exposes a separate text_config, temporarily use that text model_type.
_original_model_type = getattr(model.config, "model_type", None)
_text_model_type = None
if hasattr(model.config, "text_config"):
    _text_model_type = getattr(model.config.text_config, "model_type", None)
if _text_model_type and _text_model_type != _original_model_type:
    model.config.model_type = _text_model_type
    print(f"Temporarily swapped model.config.model_type: {_original_model_type!r} -> {_text_model_type!r}")
else:
    print(f"model.config.model_type left unchanged: {_original_model_type!r}")

try:
    trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=DPOConfig(
            beta=DPO_BETA,
            loss_type=LOSS_TYPE,
            max_length=MAX_SEQ_LENGTH,
            max_prompt_length=MAX_PROMPT_LENGTH,
            precompute_ref_log_probs=False,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_steps=warmup_steps,
            max_steps=max_steps,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            optim="adamw_8bit",
            weight_decay=0.01,
            seed=3407,
            gradient_checkpointing=True,
            dataloader_pin_memory=False,
            output_dir=str(TRAIN_DIR),
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=3,
            logging_steps=5,
            report_to="none",
            dataset_num_proc=1,
        ),
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
    )
finally:
    if _original_model_type is not None:
        model.config.model_type = _original_model_type

trainer.is_vision_model = False
print("DPO trainer configured; persistent cache handles ref logprobs.")
print(f"DPO pairs: {len(dpo_dataset):,}")
print(f"Effective batch: {effective_batch}")
print(f"Total steps: {max_steps}")
print(f"Save steps: {SAVE_STEPS}")
print("precompute_ref_log_probs: False")

## 5. Precompute Reference Log Probabilities (Persistent, Resumable Cache)

DPO needs frozen-reference log probabilities for every chosen/rejected pair. TRL's built-in `precompute_ref_log_probs=True` is one-shot and in-memory, so a kernel crash during precompute starts over from row 0.

This cell matches the other stepped DPO notebooks: it saves completed shards under `TRAIN_DIR/ref_logprobs_cache/shards/`, writes a fingerprinted manifest, resumes from existing shards, and attaches `ref_chosen_logps` / `ref_rejected_logps` to the trainer dataset before training.

In [ ]:
import hashlib
import json
import os
import time
import shutil
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_from_disk

REF_LOGPROBS_CACHE_DIR = TRAIN_DIR / "ref_logprobs_cache"
REF_LOGPROBS_SHARD_DIR = REF_LOGPROBS_CACHE_DIR / "shards"
REF_LOGPROBS_DATASET_DIR = REF_LOGPROBS_CACHE_DIR / "dataset"
REF_LOGPROBS_MANIFEST = REF_LOGPROBS_CACHE_DIR / "manifest.json"
REF_LOGPROBS_SHARD_SIZE = 64
REF_LOGPROBS_BATCH_SIZE = 1

REF_LOGPROBS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REF_LOGPROBS_SHARD_DIR.mkdir(parents=True, exist_ok=True)

_ref_cache_config = {
    "cache_version": 1,
    "base_llm": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "dpo_data_file": str(DPO_DATA_FILE),
    "dpo_data_mtime_ns": DPO_DATA_FILE.stat().st_mtime_ns if DPO_DATA_FILE.exists() else None,
    "dataset_len": len(trainer.train_dataset),
    "dataset_columns": sorted([c for c in trainer.train_dataset.column_names if not c.startswith("ref_")]),
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_vocab_size": len(tokenizer),
    "shard_size": REF_LOGPROBS_SHARD_SIZE,
    "batch_size": REF_LOGPROBS_BATCH_SIZE,
}
_ref_cache_fingerprint = hashlib.sha256(
    json.dumps(_ref_cache_config, sort_keys=True).encode("utf-8")
).hexdigest()

def _read_ref_manifest():
    if not REF_LOGPROBS_MANIFEST.exists():
        return None
    with open(REF_LOGPROBS_MANIFEST) as f:
        return json.load(f)

def _write_ref_manifest(status, completed_shards):
    manifest = {
        "status": status,
        "fingerprint": _ref_cache_fingerprint,
        "config": _ref_cache_config,
        "completed_shards": completed_shards,
        "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp_path = REF_LOGPROBS_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(manifest, f, indent=2)
    os.replace(tmp_path, REF_LOGPROBS_MANIFEST)

_manifest = _read_ref_manifest()
_cache_matches = _manifest is not None and _manifest.get("fingerprint") == _ref_cache_fingerprint
_required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}

if _cache_matches and REF_LOGPROBS_DATASET_DIR.exists():
    cached_dataset = load_from_disk(str(REF_LOGPROBS_DATASET_DIR))
    if len(cached_dataset) == len(trainer.train_dataset) and _required_ref_cols.issubset(cached_dataset.column_names):
        trainer.train_dataset = cached_dataset
        trainer._precomputed_train_ref_log_probs = True
        print(f"Loaded persistent ref logprob cache: {REF_LOGPROBS_DATASET_DIR}")
    else:
        print("Ignoring stale ref logprob dataset cache: length or columns do not match")
        _cache_matches = False

if not _required_ref_cols.issubset(trainer.train_dataset.column_names):
    if not _cache_matches:
        for stale_shard in REF_LOGPROBS_SHARD_DIR.glob("shard-*.pt"):
            stale_shard.unlink()
        _write_ref_manifest("in_progress", [])
        _manifest = _read_ref_manifest()

    num_rows = len(trainer.train_dataset)
    shard_ranges = [
        (start, min(start + REF_LOGPROBS_SHARD_SIZE, num_rows))
        for start in range(0, num_rows, REF_LOGPROBS_SHARD_SIZE)
    ]

    print("Persistent reference logprob cache")
    print(f"  Rows:       {num_rows}")
    print(f"  Shards:     {len(shard_ranges)}")
    print(f"  Shard size: {REF_LOGPROBS_SHARD_SIZE}")
    print(f"  Cache dir:  {REF_LOGPROBS_CACHE_DIR}")

    completed = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if shard_path.exists():
            completed.append(shard_idx)
            continue

        shard_dataset = trainer.train_dataset.select(range(start, end))
        shard_loader = DataLoader(
            shard_dataset,
            batch_size=REF_LOGPROBS_BATCH_SIZE,
            collate_fn=trainer.data_collator,
            num_workers=0,
            pin_memory=False,
            shuffle=False,
        )
        shard_loader = trainer.accelerator.prepare(shard_loader)

        ref_chosen_logps = []
        ref_rejected_logps = []
        for padded_batch in tqdm(shard_loader, desc=f"Ref logprobs shard {shard_idx + 1}/{len(shard_ranges)}"):
            ref_chosen_logp, ref_rejected_logp = trainer.compute_ref_log_probs(padded_batch)
            ref_chosen_logp, ref_rejected_logp = trainer.accelerator.gather_for_metrics(
                (ref_chosen_logp, ref_rejected_logp)
            )
            ref_chosen_logps.append(ref_chosen_logp.float().cpu())
            ref_rejected_logps.append(ref_rejected_logp.float().cpu())
            torch.cuda.empty_cache()
            trainer.accelerator.free_memory()

        shard_payload = {
            "fingerprint": _ref_cache_fingerprint,
            "shard_idx": shard_idx,
            "start": start,
            "end": end,
            "ref_chosen_logps": torch.cat(ref_chosen_logps),
            "ref_rejected_logps": torch.cat(ref_rejected_logps),
        }
        tmp_shard_path = shard_path.with_suffix(".pt.tmp")
        torch.save(shard_payload, tmp_shard_path)
        os.replace(tmp_shard_path, shard_path)
        completed.append(shard_idx)
        _write_ref_manifest("in_progress", completed)

    all_ref_chosen_logps = []
    all_ref_rejected_logps = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if not shard_path.exists():
            raise RuntimeError(f"Missing ref logprob shard: {shard_path}")
        shard_payload = torch.load(shard_path, map_location="cpu")
        if shard_payload.get("fingerprint") != _ref_cache_fingerprint:
            raise RuntimeError(f"Stale ref logprob shard fingerprint: {shard_path}")
        if shard_payload.get("start") != start or shard_payload.get("end") != end:
            raise RuntimeError(f"Ref logprob shard range mismatch: {shard_path}")
        all_ref_chosen_logps.append(shard_payload["ref_chosen_logps"])
        all_ref_rejected_logps.append(shard_payload["ref_rejected_logps"])

    ref_chosen_values = torch.cat(all_ref_chosen_logps).numpy()
    ref_rejected_values = torch.cat(all_ref_rejected_logps).numpy()
    if len(ref_chosen_values) != num_rows or len(ref_rejected_values) != num_rows:
        raise RuntimeError("Ref logprob cache length does not match training dataset")

    train_dataset_with_ref = trainer.train_dataset
    for ref_col in ["ref_chosen_logps", "ref_rejected_logps"]:
        if ref_col in train_dataset_with_ref.column_names:
            train_dataset_with_ref = train_dataset_with_ref.remove_columns(ref_col)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_chosen_logps", ref_chosen_values)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_rejected_logps", ref_rejected_values)

    if REF_LOGPROBS_DATASET_DIR.exists():
        shutil.rmtree(REF_LOGPROBS_DATASET_DIR)
    train_dataset_with_ref.save_to_disk(str(REF_LOGPROBS_DATASET_DIR))
    trainer.train_dataset = train_dataset_with_ref
    trainer._precomputed_train_ref_log_probs = True
    _write_ref_manifest("complete", list(range(len(shard_ranges))))
    print(f"Saved persistent ref logprob dataset cache: {REF_LOGPROBS_DATASET_DIR}")

print(f"Ref logprob columns ready: {sorted(_required_ref_cols)}")

## 6. Train

This cell should run after the persistent reference-logprob cache cell. It uses normal trainer checkpoints every `SAVE_STEPS` steps and resumes from the latest checkpoint if one exists.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}
if not required_ref_cols.issubset(trainer.train_dataset.column_names):
    raise RuntimeError("Run the reference logprob cache cell before training.")

print("DPO Training started...")
print("Watch for loss to decrease from about 0.69 toward 0.40-0.55.")
print("Reward margins should widen (chosen > rejected).\n")

last_checkpoint = get_last_checkpoint(trainer.args.output_dir)
if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    result = trainer.train()

print("\nDPO training complete")
print(f"Final loss: {result.training_loss:.4f}")
print(f"Total steps: {result.global_step}")

## 7. Save DPO LoRA Adapter

Save the DPO adapter and write a `complete.json` sentinel with the last checkpoint and ref-logprob cache fingerprint.

In [ ]:
import json, shutil, time
from transformers.trainer_utils import get_last_checkpoint

LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

persona_prompts = SFT_LORA_PATH / "persona_system_prompts.json"
if persona_prompts.exists():
    shutil.copy2(persona_prompts, LORA_OUTPUT_DIR / "persona_system_prompts.json")

metadata = {
    "status": "complete",
    "model_name_base": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "dpo_data_file": str(DPO_DATA_FILE),
    "train_dir": str(TRAIN_DIR),
    "last_checkpoint": get_last_checkpoint(str(TRAIN_DIR)),
    "global_step": getattr(trainer.state, "global_step", None),
    "max_steps": getattr(trainer.state, "max_steps", None),
    "dpo_beta": DPO_BETA,
    "learning_rate": LEARNING_RATE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "save_steps": SAVE_STEPS,
    "ref_logprob_cache_fingerprint": globals().get("_ref_cache_fingerprint"),
    "saved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(LORA_OUTPUT_DIR / "dpo_training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
with open(LORA_OUTPUT_DIR / "complete.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved DPO LoRA to {LORA_OUTPUT_DIR}")
print(f"Saved completion sentinel to {LORA_OUTPUT_DIR / 'complete.json'}")

## 8. Quick Evaluation

In [ ]:
from transformers import TextStreamer
FastLanguageModel.for_inference(model)
eval_prompts = [("voice discipline", "You are Isaiah, a court prophet who beheld God's throne and heard the seraphim cry 'Holy, holy, holy.' Speak in first person from your lived prophetic witness.", "What should I understand about forgiveness when I feel unworthy?"), ("scripture fabrication resistance", "You are Paul, apostle to the Gentiles. Speak from your letters and lived witness.", "Where exactly does Scripture say that angels taught Paul the doctrine of justification?"), ("depth over platitude", "You are Job, speaking from the ash heap and wrestling honestly before God.", "Why does God allow righteous people to suffer?")]
for name, system_prompt, user_prompt in eval_prompts:
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    print(f"\n=== {name.upper()} ===")
    print(f"USER: {user_prompt}")
    print("MODEL: ", end="")
    _ = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, streamer=TextStreamer(tokenizer, skip_prompt=True))